In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation


In [2]:
train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')
train_df.head(2), test_df.head(2)

(   Index geohash  day  timestamp    demand     RoadType  NumberofLanes  \
 0      0  qp02z1   48        0.0  0.048804          NaN              1   
 1      1  qp02zt   48        0.0  0.118507  Residential              3   
 
   LargeVehicles Landmarks  Temperature Weather  
 0   Not Allowed        No          NaN     NaN  
 1       Allowed       Yes    31.104565   Sunny  ,
    Index geohash  day timestamp     RoadType  NumberofLanes LargeVehicles  \
 0      0  qp02z1   49      2:15          NaN              1   Not Allowed   
 1      1  qp02z9   49      2:15  Residential              1   Not Allowed   
 
   Landmarks  Temperature Weather  
 0        No          NaN     NaN  
 1        No     6.476213   Snowy  )

In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  float64
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(3), int64(3), object(5)
memory usage: 6.5+ MB


In [4]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41454 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    40429 non-null  float64
 9   Weather        41347 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB


In [5]:
train_df.isnull().sum(), test_df.isnull().sum()

(Index               0
 geohash             0
 day                 0
 timestamp           0
 demand              0
 RoadType          600
 NumberofLanes       0
 LargeVehicles       0
 Landmarks           0
 Temperature      2495
 Weather           797
 dtype: int64,
 Index               0
 geohash             0
 day                 0
 timestamp           0
 RoadType          324
 NumberofLanes       0
 LargeVehicles       0
 Landmarks           0
 Temperature      1349
 Weather           431
 dtype: int64)

In [6]:
train_df.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
       'Weather'],
      dtype='object')

In [7]:
def unique_elm(df):
    for col in df.columns:
        print('--------------------------------------------')
        lt = df[col].unique()
        print(col, " :  total unique : ", len(lt))
        print(lt)

In [30]:
unique_elm(train_df)

--------------------------------------------
Index  :  total unique :  77299
[    0     1     2 ... 77296 77297 77298]
--------------------------------------------
geohash  :  total unique :  1249
['qp02z1' 'qp02zt' 'qp08bj' ... 'qp098v' 'qp0d5c' 'qp0d55']
--------------------------------------------
day  :  total unique :  2
[48 49]
--------------------------------------------
timestamp  :  total unique :  96
[ 0.    0.15  0.3   0.45  1.    1.15  1.3   1.45  2.    2.15  2.3   2.45
  3.    3.15  3.3   3.45  4.    4.15  4.3   4.45  5.    5.15  5.3   5.45
  6.    6.15  6.3   6.45  7.    7.15  7.3   7.45  8.    8.15  8.3   8.45
  9.    9.15  9.3   9.45 10.   10.15 10.3  10.45 11.   11.15 11.3  11.45
 12.   12.15 12.3  12.45 13.   13.15 13.3  13.45 14.   14.15 14.3  14.45
 15.   15.15 15.3  15.45 16.   16.15 16.3  16.45 17.   17.15 17.3  17.45
 18.   18.15 18.3  18.45 19.   19.15 19.3  19.45 20.   20.15 20.3  20.45
 21.   21.15 21.3  21.45 22.   22.15 22.3  22.45 23.   23.15 23.3  23.45]
-

In [31]:
unique_elm(test_df)

--------------------------------------------
Index  :  total unique :  41778
[    0     1     2 ... 41775 41776 41777]
--------------------------------------------
geohash  :  total unique :  1190
['qp02z1' 'qp02z9' 'qp02yf' ... 'qp08cv' 'qp09kq' 'qp0dhu']
--------------------------------------------
day  :  total unique :  1
[49]
--------------------------------------------
timestamp  :  total unique :  47
['2:15' '2:30' '2:45' '3:0' '3:15' '3:30' '3:45' '4:0' '4:15' '4:30'
 '4:45' '5:0' '5:15' '5:30' '5:45' '6:0' '6:15' '6:30' '6:45' '7:0' '7:15'
 '7:30' '7:45' '8:0' '8:15' '8:30' '8:45' '9:0' '9:15' '9:30' '9:45'
 '10:0' '10:15' '10:30' '10:45' '11:0' '11:15' '11:30' '11:45' '12:0'
 '12:15' '12:30' '12:45' '13:0' '13:15' '13:30' '13:45']
--------------------------------------------
RoadType  :  total unique :  4
[nan 'Residential' 'Street' 'Highway']
--------------------------------------------
NumberofLanes  :  total unique :  5
[1 3 2 5 4]
-----------------------------------------

In [33]:
lt = (train_df['geohash'].str[:4]).unique()
lt, len(lt)

(array(['qp02', 'qp08', 'qp03', 'qp09', 'qp06', 'qp0d'], dtype=object), 6)

In [34]:
lt = (test_df['geohash'].str[:4]).unique()
lt, len(lt)

(array(['qp02', 'qp08', 'qp03', 'qp09', 'qp06', 'qp0d'], dtype=object), 6)

### TimeStamp Feature Fix

In [8]:
# Converts training floats (e.g., 2.15 -> "02:15", 2.3 -> "02:30") to matching string formats
def convert_train_timestamp(x):
    hour = int(x)
    minute = int(round((x - hour) * 100))
    return f"{hour:02d}:{minute:02d}"

print(convert_train_timestamp(2.15))  # Output: "02:15"

02:15


In [9]:
train_df['time_str'] = train_df['timestamp'].apply(convert_train_timestamp)
test_df['time_str'] = test_df['timestamp'].astype(str)

#### combine to preprocess

In [47]:
train_df['is_train'] = 1
test_df['is_train'] = 0
test_df['demand'] = np.nan

full_df = pd.concat([train_df, test_df], ignore_index=True)


In [48]:
# Generate uniform datetime features

full_df['dt'] = pd.to_datetime(full_df['time_str'], format='%H:%M')

full_df['hour'] = full_df['dt'].dt.hour
full_df['minute'] = full_df['dt'].dt.minute
full_df['time_in_minutes'] = full_df['hour'] * 60 + full_df['minute']


In [49]:
# drop unused columns
full_df.drop(columns=['timestamp', 'time_str', 'dt'], inplace=True)


### Fill Missing Values

In [50]:
full_df.isnull().sum()

Index                  0
geohash                0
day                    0
demand             41778
RoadType             924
NumberofLanes          0
LargeVehicles          0
Landmarks              0
Temperature         3844
Weather             1228
is_train               0
hour                   0
minute                 0
time_in_minutes        0
dtype: int64

In [51]:
full_df['Temperature'] = full_df['Temperature'].fillna(full_df['Temperature'].median())

full_df['RoadType'] = full_df['RoadType'].fillna('Unknown')
full_df['Weather'] = full_df['Weather'].fillna('Unknown')

# fill categorical columns with the most frequent value:
# train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
# train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])

### Feature Engineering

In [38]:
full_df['day'].unique()

array([48, 49])

In [39]:
lt = (full_df['geohash'].str[:4]).unique()
lt, len(lt)

(array(['qp02', 'qp08', 'qp03', 'qp09', 'qp06', 'qp0d'], dtype=object), 6)

In [35]:
lt = full_df['geohash'].str[:5].unique()
lt, len(lt)


(array(['qp02z', 'qp08b', 'qp08g', 'qp08c', 'qp08f', 'qp03p', 'qp090',
        'qp094', 'qp095', 'qp09h', 'qp09j', 'qp091', 'qp03j', 'qp03n',
        'qp03m', 'qp03q', 'qp03r', 'qp092', 'qp093', 'qp096', 'qp097',
        'qp09k', 'qp09m', 'qp03t', 'qp03w', 'qp03x', 'qp098', 'qp099',
        'qp09d', 'qp09e', 'qp09s', 'qp09t', 'qp09w', 'qp03y', 'qp03z',
        'qp09b', 'qp09g', 'qp09u', 'qp09v', 'qp03v', 'qp09c', 'qp09f',
        'qp09y', 'qp06n', 'qp06p', 'qp0d0', 'qp0d1', 'qp0d4', 'qp0d5',
        'qp0dh', 'qp0dj', 'qp0dn', 'qp02y', 'qp08u', 'qp09q', 'qp09n'],
       dtype=object),
 56)

In [52]:
full_df['is_day_49'] = (full_df['day'] == 49).astype(int)

full_df['geohash_zone_4'] = full_df['geohash'].str[:4]
full_df['geohash_zone_5'] = full_df['geohash'].str[:5]



##### convert time_in_minutes into a cyclic circl

In [ ]:
time_in_minutes = 24*60
time_in_minutes
# 23:59 --> 00:00 are in cyclic

1440

In [54]:
# 1440 minutes in a day
full_df['time_sin'] = np.sin(2 * np.pi * full_df['time_in_minutes'] / 1440)
full_df['time_cos'] = np.cos(2 * np.pi * full_df['time_in_minutes'] / 1440)

#### avg demand per exact geohash

In [55]:
train_mask = full_df['is_train'] == 1
global_mean = full_df.loc[train_mask, 'demand'].mean()
global_mean

np.float64(0.09394238120627693)

In [56]:
# Average demand per exact geohash
geohash_map = full_df[train_mask].groupby('geohash')['demand'].mean().to_dict()
full_df['geohash_hist_demand'] = full_df['geohash'].map(geohash_map).fillna(global_mean)

In [57]:
# Average demand per broader zone 
zone4_map = full_df[train_mask].groupby('geohash_zone_4')['demand'].mean().to_dict()
full_df['zone4_hist_demand'] = full_df['geohash_zone_4'].map(zone4_map).fillna(global_mean)

### convert Data type

In [42]:
full_df.columns

Index(['Index', 'geohash', 'day', 'demand', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'is_train',
       'hour', 'minute', 'time_in_minutes', 'is_day_49', 'geohash_zone_4',
       'geohash_zone_5', 'time_sin', 'time_cos'],
      dtype='object')

In [58]:
lt = full_df.select_dtypes(include='object').columns
lt, len(lt)

(Index(['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
        'geohash_zone_4', 'geohash_zone_5'],
       dtype='object'),
 7)

In [59]:
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash_zone_4', 'geohash_zone_5']
for col in categorical_cols:
    full_df[col] = full_df[col].astype('category')


### Separte Train and test

In [60]:
train_clean = full_df[full_df['is_train'] == 1].drop(columns=['is_train', 'Index'])
test_clean = full_df[full_df['is_train'] == 0].drop(columns=['is_train', 'demand'])
test_indices = full_df[full_df['is_train'] == 0]['Index']

full_df.shape, train_clean.shape, test_clean.shape

((119077, 21), (77299, 19), (41778, 19))

In [61]:
X = train_clean.drop(columns=['demand'])
y = train_clean['demand']
X.shape, y.shape

((77299, 18), (77299,))

### Train

In [62]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [63]:
model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=63,             # parameter for LightGBM complexity
    max_depth=8,               # structural tree depth 
    min_child_samples=30,      # Prevents splits on tiny sample paths
    subsample=0.8,             # Row bagging fraction
    colsample_bytree=0.8,      # feature bagging fraction
    random_state=42,
    verbose=50,
    n_jobs=-1                  # Employs all CPU cores for training speed
)

In [64]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[
        early_stopping(stopping_rounds=50, verbose=True),
        log_evaluation(period=100)
    ]
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.896867
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.223627
[LightGBM] [Debug] init for col-wise cost 0.003205 seconds, init for row-wise cost 0.008745 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004804 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Dense Multi-Val Bin
[LightGBM] [Info] Total Bins 1943
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 18
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Debug] Trained a tree with leaves = 63 and depth 

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,8
,learning_rate,0.03
,n_estimators,1500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,30


### Evaluate performance

In [65]:
val_preds = model.predict(X_val)
print(f"\nFinal Validation LightGBM R2 Score: {r2_score(y_val, val_preds):.5f}")


Final Validation LightGBM R2 Score: 0.96283


### Train without split train data for submission

##### K-Fold Cross Validation

In [66]:
from sklearn.model_selection import KFold

# Setup K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_predictions = np.zeros(len(test_clean))
val_scores = []

test_features = test_clean.drop(columns=['Index'])

In [67]:
# . Train 5 models
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Training Fold {fold+1} ---")
    
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # Initialize a fresh model for this fold
    model = LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=8,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42 + fold, # Different seed per fold for diversity
        n_jobs=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Evaluate this fold
    val_preds = model.predict(X_va)
    score = r2_score(y_va, val_preds)
    val_scores.append(score)
    print(f"Fold {fold+1} R2 Score: {score:.5f}")
    
    # Add 1/5th of the test predictions to the final array
    test_predictions += model.predict(test_features) / 5.0


--- Training Fold 1 ---
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.896867
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.223627
[LightGBM] [Debug] init for col-wise cost 0.003178 seconds, init for row-wise cost 0.009984 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011535 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1943
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 18
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Debug] Trained a tree with leaves = 63 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 63 and depth = 8
[LightGBM] [Debu

In [68]:

print(f"\nAverage Cross-Validation R2: {np.mean(val_scores):.5f}")

# Generate the Final Submission
submission = pd.DataFrame({
    'Index': test_indices,
    'demand': test_predictions # This is now the average of 5 highly optimized models
})
submission.to_csv('submission_lgb_kfold.csv', index=False)


Average Cross-Validation R2: 0.96229
